### Requirements:

1. Dataset ko properly load karein aur *EDA (Exploratory Data Analysis)* perform karein.
2. Dataset mein:

   * Missing values
   * Duplicate values
   * Outliers
   * Target class distribution
     check karein.
3. Features (X) aur target (y) ko properly separate karein.
4. Categorical columns ko suitable *Encoding Technique* ke through convert karein.
5. Date-related column customer_since ko zarurat ke mutabiq suitable format/features mein convert karein.
6. Dataset ko *Training aur Testing data* mein divide karein.
7. Agar target classes imbalanced hon to *SMOTE* ya koi suitable balancing technique apply karein.
8. Suitable *Feature Scaling* technique apply karein, jaise:

   * StandardScaler
   * MinMaxScaler

   Scaling technique khud select karein aur explain karein ke aapne woh technique kyun choose ki.
9. Neeche diye gaye *Classification Algorithms lazmi apply karein*:

   * Decision Tree
   * Random Forest
   * XGBoost

   Iske ilawa aap koi aur suitable classification algorithm bhi apply kar sakte hain.
10. Har model ki performance evaluate karein using:

* Accuracy
* Precision
* Recall
* F1-Score
* Confusion Matrix
* Classification Report

11. Sabhi models ki performance ko compare karein.
12. *Best-performing model identify karein*.
13. Clearly mention karein:

* Kaunsi encoding technique use ki aur kyun.
* Kaunsi scaling technique use ki aur kyun.
* SMOTE/balancing technique use ki ya nahi aur kyun.
* Decision Tree ki accuracy/performance.
* Random Forest ki accuracy/performance.
* XGBoost ki accuracy/performance.
* Kaunsa model best perform kiya.
* Best model ki final performance kya rahi.

### Important:

Aapko sirf models apply nahi karne. Har preprocessing step aur model selection ko *properly explain* karna hai.

Especially ye explain karna zaroori hai ke:

*Data → Preprocessing → Encoding → Train/Test Split → Scaling → SMOTE (if required) → Models → Evaluation → Model Comparison → Best Model*

In [ ]:
!pip install xgboost

In [135]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LassoCV, RidgeCV
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

# Data

In [28]:
# loading dataset
data = pd.read_csv('customer_analytics_dataset.csv')

In [29]:
data.head()

,customer_id,age,gender,country,avg_order_value,total_orders,last_purchase,is_fraudulent,preferred_category,email_open_rate,customer_since,loyalty_score,churn_risk
0,CUST_8270,30,Female,Brazil,101.08,8,176,1,Beauty,25.6,2024-06-05,50,0.20
1,CUST_1860,53,Female,USA,90.39,10,88,0,Electronics,12.3,2024-02-19,37,0.34
2,CUST_6390,73,Male,Australia,83.28,6,203,0,Sports,NaN,2024-04-16,65,0.05
3,CUST_6191,30,Other,Japan,109.90,9,346,1,Electronics,42.9,2020-07-08,93,0.19
4,CUST_6734,29,Female,Canada,269.38,16,342,0,Fashion,5.3,2025-04-09,79,0.15


In [30]:
data = data.drop(columns=['customer_id'])

In [31]:
data.shape

(5000, 12)

In [32]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   age                 5000 non-null   int64  
 1   gender              5000 non-null   str    
 2   country             5000 non-null   str    
 3   avg_order_value     4750 non-null   float64
 4   total_orders        5000 non-null   int64  
 5   last_purchase       5000 non-null   int64  
 6   is_fraudulent       5000 non-null   int64  
 7   preferred_category  5000 non-null   str    
 8   email_open_rate     4750 non-null   float64
 9   customer_since      5000 non-null   str    
 10  loyalty_score       5000 non-null   int64  
 11  churn_risk          5000 non-null   float64
dtypes: float64(3), int64(5), str(4)
memory usage: 601.8 KB


# EDA

In [56]:
# checking duplicate values
data.duplicated().sum()

np.int64(0)

In [33]:
# removing null values
data.isna().sum()

age                     0
gender                  0
country                 0
avg_order_value       250
total_orders            0
last_purchase           0
is_fraudulent           0
preferred_category      0
email_open_rate       250
customer_since          0
loyalty_score           0
churn_risk              0
dtype: int64

In [34]:
data.avg_order_value.nunique()

4221

In [35]:
data['avg_order_value'] = data.avg_order_value.fillna(data.avg_order_value.mean())

In [36]:
data.email_open_rate.nunique()

989

In [37]:
data['email_open_rate'] = data.email_open_rate.fillna(data.email_open_rate.mean())

In [59]:
data.gender.unique()

<ArrowStringArray>
['Female', 'Male', 'Other']
Length: 3, dtype: str

In [60]:
data.country.unique()

<ArrowStringArray>
[   'Brazil',       'USA', 'Australia',     'Japan',    'Canada',    'France',
     'India',     'China',   'Germany',        'UK']
Length: 10, dtype: str

In [61]:
data.preferred_category.unique()

<ArrowStringArray>
['Beauty', 'Electronics', 'Sports', 'Fashion', 'Home']
Length: 5, dtype: str

In [51]:
# replacing outliers by mean
def remove_outlier(data):
    for i in data.select_dtypes(exclude=['object']).columns:
        if data[i].nunique() <= 2:
            continue
        q1 = np.percentile(data[i], 25)
        q3 = np.percentile(data[i], 75)
        iqr = q3 - q1
        lower_bound = q1 - (1.5 * iqr)
        upper_bound = q3 + (1.5 * iqr)
        condition = (data[i]>upper_bound) | (data[i]<lower_bound)
        data.loc[condition, i] = int(data[i].mean())
    return 'Outlier has been removed!'

In [52]:
# removing outliers
remove_outlier(data)

'Outlier has been removed!'

In [57]:
# checking target class distribution 
data.is_fraudulent.value_counts()

is_fraudulent
0    4871
1     129
Name: count, dtype: int64

In [74]:
# handling Date
date = pd.to_datetime(data.customer_since)
data['customer_since_year'] = date.dt.year
data['customer_since_month'] = date.dt.month
data['customer_since_day'] = date.dt.day
data.drop(columns=['customer_since'], axis= 0, inplace= True)

In [143]:
# Seperating input/output
X = data.drop(columns=['is_fraudulent'])
y = data.is_fraudulent

# Encoding

In [144]:
# apply onehotencoding 
encoding = OneHotEncoder(sparse_output=False)
cat_cols = ['gender', 'country', 'preferred_category']
encoded_array = encoding.fit_transform(X[cat_cols])
encoded_df = pd.DataFrame(data= encoded_array, columns= encoding.get_feature_names_out(cat_cols), index= X.index)
X = X.drop(columns=cat_cols).join(encoded_df)

# Spliting training testing data

In [145]:
# spliting training testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [154]:
smote = SMOTE()
X_train, y_train = smote.fit_resample(X_train, y_train)
y_train.value_counts()

is_fraudulent
0    3897
1    3897
Name: count, dtype: int64

# Scaling

In [155]:
# MinMax scaling
scale = MinMaxScaler()
X_train = scale.fit_transform(X_train)
X_test = scale.transform(X_test)

# Models

In [156]:
# logistic regression
lr = LogisticRegression(random_state= 42)
lr.fit(X_train, y_train)
y_pred_train_lr = lr.predict(X_train)
y_pred_test_lr = lr.predict(X_test)
print(classification_report(y_train, y_pred_train_lr))
print(classification_report(y_test, y_pred_test_lr))
print('Train', confusion_matrix(y_train, y_pred_train_lr))
print('Test', confusion_matrix(y_test, y_pred_test_lr))

              precision    recall  f1-score   support

           0       0.67      0.64      0.65      3897
           1       0.65      0.68      0.67      3897

    accuracy                           0.66      7794
   macro avg       0.66      0.66      0.66      7794
weighted avg       0.66      0.66      0.66      7794

              precision    recall  f1-score   support

           0       0.97      0.63      0.76       974
           1       0.02      0.31      0.04        26

    accuracy                           0.62      1000
   macro avg       0.50      0.47      0.40      1000
weighted avg       0.95      0.62      0.74      1000

Train [[2497 1400]
 [1252 2645]]
Test [[610 364]
 [ 18   8]]


In [157]:
# support vector classification
svc = SVC(kernel= 'rbf' ,random_state= 42)
svc.fit(X_train, y_train)
y_pred_train_svc = svc.predict(X_train)
y_pred_test_svc = svc.predict(X_test)
print(classification_report(y_train, y_pred_train_svc))
print(classification_report(y_test, y_pred_test_svc))
print('Train', confusion_matrix(y_train, y_pred_train_svc))
print('Test', confusion_matrix(y_test, y_pred_test_svc))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3897
           1       0.99      0.99      0.99      3897

    accuracy                           0.99      7794
   macro avg       0.99      0.99      0.99      7794
weighted avg       0.99      0.99      0.99      7794

              precision    recall  f1-score   support

           0       0.97      0.98      0.98       974
           1       0.00      0.00      0.00        26

    accuracy                           0.95      1000
   macro avg       0.49      0.49      0.49      1000
weighted avg       0.95      0.95      0.95      1000

Train [[3843   54]
 [  53 3844]]
Test [[954  20]
 [ 26   0]]


In [166]:
# decision tree classifier
dt = DecisionTreeClassifier(random_state= 42, criterion= 'gini')
dt.fit(X_train, y_train)
y_pred_train_dt = dt.predict(X_train)
y_pred_test_dt = dt.predict(X_test)
print(classification_report(y_train, y_pred_train_dt))
print(classification_report(y_test, y_pred_test_dt))
print('Train', confusion_matrix(y_train, y_pred_train_dt))
print('Test', confusion_matrix(y_test, y_pred_test_dt))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3897
           1       1.00      1.00      1.00      3897

    accuracy                           1.00      7794
   macro avg       1.00      1.00      1.00      7794
weighted avg       1.00      1.00      1.00      7794

              precision    recall  f1-score   support

           0       0.97      0.97      0.97       974
           1       0.03      0.04      0.03        26

    accuracy                           0.94      1000
   macro avg       0.50      0.50      0.50      1000
weighted avg       0.95      0.94      0.95      1000

Train [[3897    0]
 [   0 3897]]
Test [[942  32]
 [ 25   1]]


In [159]:
# random forest classification
rf = RandomForestClassifier(n_estimators= 100,class_weight= 'balanced', random_state=42, criterion= 'gini')
rf.fit(X_train, y_train)
y_pred_train_rf = rf.predict(X_train)
y_pred_test_rf = rf.predict(X_test)
print(classification_report(y_train, y_pred_train_rf))
print(classification_report(y_test, y_pred_test_rf))
print('Train', confusion_matrix(y_train, y_pred_train_rf))
print('Test', confusion_matrix(y_test, y_pred_test_rf))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3897
           1       1.00      1.00      1.00      3897

    accuracy                           1.00      7794
   macro avg       1.00      1.00      1.00      7794
weighted avg       1.00      1.00      1.00      7794

              precision    recall  f1-score   support

           0       0.97      1.00      0.99       974
           1       0.00      0.00      0.00        26

    accuracy                           0.97      1000
   macro avg       0.49      0.50      0.49      1000
weighted avg       0.95      0.97      0.96      1000

Train [[3897    0]
 [   1 3896]]
Test [[974   0]
 [ 26   0]]


C:\Users\S.M. Abdullah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\S.M. Abdullah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\S.M. Abdullah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [160]:
# XG boost
xgb = XGBClassifier(max_depth=3, learning_rate=0.05, n_estimators=100, random_state=42)
xgb.fit(X_train, y_train)
y_pred_train_xgb = xgb.predict(X_train)
y_pred_test_xgb = xgb.predict(X_test)
print(classification_report(y_train, y_pred_train_xgb))
print(classification_report(y_test, y_pred_test_xgb))
print('Train', confusion_matrix(y_train, y_pred_train_xgb))
print('Test', confusion_matrix(y_test, y_pred_test_xgb))

              precision    recall  f1-score   support

           0       0.95      1.00      0.98      3897
           1       1.00      0.95      0.98      3897

    accuracy                           0.98      7794
   macro avg       0.98      0.98      0.98      7794
weighted avg       0.98      0.98      0.98      7794

              precision    recall  f1-score   support

           0       0.97      1.00      0.99       974
           1       0.00      0.00      0.00        26

    accuracy                           0.97      1000
   macro avg       0.49      0.50      0.49      1000
weighted avg       0.95      0.97      0.96      1000

Train [[3897    0]
 [ 184 3713]]
Test [[974   0]
 [ 26   0]]


C:\Users\S.M. Abdullah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\S.M. Abdullah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\S.M. Abdullah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [167]:
# Comparing Accuracy
model_name = ['LogisticRegression', 'SVC', 'DecisionTreeClassifier', 'RandomForestClassifier', 'XGBClasifier']
train = [accuracy_score(y_train, y_pred_train_lr), accuracy_score(y_train, y_pred_train_svc), accuracy_score(y_train, y_pred_train_dt), accuracy_score(y_train, y_pred_train_rf), accuracy_score(y_train, y_pred_train_xgb)]
test = [accuracy_score(y_test, y_pred_test_lr), accuracy_score(y_test, y_pred_test_svc), accuracy_score(y_test, y_pred_test_dt), accuracy_score(y_test, y_pred_test_rf), accuracy_score(y_test, y_pred_test_xgb)]
compare = pd.DataFrame({'Model': model_name, 'Train Accuracy': train, 'Test Accuracy': test})
compare

,Model,Train Accuracy,Test Accuracy
0,LogisticRegression,0.659738,0.618
1,SVC,0.986271,0.954
2,DecisionTreeClassifier,1.000000,0.943
3,RandomForestClassifier,0.999872,0.974
4,XGBClasifier,0.976392,0.974


## Accuracy basis par XGBoost (97.4%) best perform kar raha hai, lekin severe class imbalance hone ki wajah se F1-score aur Recall ko evaluate karna ziada critical hai.